In [1]:
# %%
import time
import os
import json
import random
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    run_all_pairwise_projections_parallel,
    generate_pairwise_comparisons_from_df
)

# %% 
# --- 1) Pfade & Daten einlesen ---
data_path = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits/eurasian_otter_(1)")
df = pd.read_csv(data_path / "train.csv")

# %% 
# --- 2) Feature-Spalten und Parameter definieren ---
feature_cols = [
    col for col in df.columns
    if col.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[col])
]

feature_counts      = [10, 15, 20, 50, 100, 200]
selection_methods   = ["random_forest"]
reducers            = ["pca", "lda", "umap"]
n_components_list   = [2, 3, 4]
n_jobs              = -1

output_dir = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data")
output_dir.mkdir(parents=True, exist_ok=True)

# %%
# --- 3) Comparisons & Summary erzeugen ---
comparisons, summary = generate_pairwise_comparisons_from_df(
    df,
    id_col="individual_id",
    chunk_size=7,
    trail_size_list=[7, 5, 3],
    max_individuals=3,
    max_trails_per_animal=5,
    n_folds=5,
    random_state=42
)

# %%
# --- 4) Summary anzeigen ---
print("=== Summary pro Trail-Länge ===")
display(summary)

# %%
n_total = len(comparisons)
n_same  = sum(1 for comp in comparisons if comp.get("same_individual"))
n_diff  = n_total - n_same

print(f"→ Insgesamt {n_total} Vergleichspaare erzeugt.")
print(f"   davon {n_same} Paare mit gleichem Individuum, {n_diff} Paare mit unterschiedlichem Individuum.\n")

print("=== Beispiel-Einträge ===")
for comp in comparisons[:5]:
    print(json.dumps(comp, indent=2, sort_keys=True))

# %%
# --- 6) Sweep über FS-Methoden und Sex-Model ein/aus ---
all_results = []
timing_log  = []

for sel in tqdm(selection_methods, desc="FS method"):
    t_sel = time.time()
    for use_sex in (False, True):
        tag = "sex_on" if use_sex else "sex_off"
        with tqdm_joblib(tqdm(desc=f"Pairs ({sel} | {tag})", total=len(comparisons))):
            results = run_all_pairwise_projections_parallel(
                comparisons=comparisons,
                df=df,
                feature_cols=feature_cols,
                k_features=feature_counts,
                reducers=reducers,
                selection_method=sel,
                n_components=n_components_list,
                use_sexmodel_prediction=use_sex,
                sexmodel_path=(
                    r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\models\eurasian_otter_(1).joblib"
                    if use_sex else None
                ),
                debug=False,
                n_jobs=n_jobs
            )
        dur = time.time() - t_sel
        print(f"→ {sel} | sex={use_sex}: {len(results)} rows in {dur:.1f}s")

        df_res = pd.DataFrame(results)
        df_res["selection_method"] = sel
        df_res["use_sexmodel_prediction"] = use_sex
        fname = f"pairwise_{sel}_{tag}_{len(comparisons)}pairs.csv"
        df_res.to_csv(output_dir / fname, index=False)

        all_results.append(df_res)
        timing_log.append({
            "selection_method": sel,
            "use_sexmodel_prediction": use_sex,
            "duration_s": dur,
            "n_results": len(results)
        })

# %%
# --- 7) Timing & Gesamt-Ergebnis speichern ---
pd.DataFrame(timing_log).to_csv(output_dir / "timing_log.csv", index=False)

if all_results:
    full = pd.concat(all_results, ignore_index=True)
    full.to_csv(output_dir / "all_results_combined.csv", index=False)
    display(full.head())
else:
    print("Keine Ergebnisse zum Zusammenführen.")


c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


=== Summary pro Trail-Länge ===


,sub_size,n_animals,n_trails,avg_comp_per_ind_same,sd_comp_per_ind_same,avg_comp_per_ind_diff,sd_comp_per_ind_diff,ratio_same_to_diff
0,7,3,7,NaN,NaN,NaN,NaN,NaN
1,5,3,7,NaN,NaN,NaN,NaN,NaN
2,3,3,7,NaN,NaN,NaN,NaN,NaN
3,Total,3,63,5.0,3.464102,32.0,3.464102,0.3125


→ Insgesamt 63 Vergleichspaare erzeugt.
   davon 15 Paare mit gleichem Individuum, 48 Paare mit unterschiedlichem Individuum.

=== Beispiel-Einträge ===
{
  "chunk_a": 0,
  "chunk_b": 0,
  "diff_size": 0,
  "fold": 0,
  "ind_a": "mette",
  "ind_b": "rosie",
  "same_individual": false,
  "samples_a": [
    266,
    261,
    268,
    262,
    269,
    258,
    264
  ],
  "samples_b": [
    29,
    31,
    30,
    49,
    38,
    40,
    41
  ],
  "trail_size_a": 7,
  "trail_size_b": 7
}
{
  "chunk_a": 0,
  "chunk_b": 1,
  "diff_size": 0,
  "fold": 4,
  "ind_a": "mette",
  "ind_b": "rosie",
  "same_individual": false,
  "samples_a": [
    266,
    261,
    268,
    262,
    269,
    258,
    264
  ],
  "samples_b": [
    45,
    35,
    39,
    43,
    42,
    27,
    48
  ],
  "trail_size_a": 7,
  "trail_size_b": 7
}
{
  "chunk_a": 0,
  "chunk_b": 2,
  "diff_size": 0,
  "fold": 3,
  "ind_a": "mette",
  "ind_b": "rosie",
  "same_individual": false,
  "samples_a": [
    266,
    261,
    2

FS method:   0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/63 [00:00<?, ?it/s]

Processing Pairs:   0%|          | 0/63 [00:12<?, ?it/s]




  0%|          | 0/63 [00:00<?, ?it/s]

→ random_forest | sex=True: 0 rows in 12.5s


,selection_method,use_sexmodel_prediction


In [ ]:
# %% 
import time
import os
import pandas as pd
import random
import json
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    run_all_pairwise_projections_parallel,
    generate_pairwise_comparisons_from_df
)

# --- Basis-Pfade ---
splits_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits")
models_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/models")
output_root = Path("D:/Projects and Disschaptors/FIT_package/results/data")
output_root.mkdir(parents=True, exist_ok=True)

# --- Gemeinsame Parameter ---
feature_counts     = [10,12,16, 15,16, 20, 50, 100]
selection_methods  = ["random_forest","forward"]
reducers           = ["pca", "lda", "umap"]
n_components_list  = [2, 3, 4]
n_jobs             = -1

# Zum globalen Zusammenbau
all_species_results = []

# %%  
# Schleife über alle Species-Ordner
for species_dir in sorted(splits_root.iterdir()):
    if not species_dir.is_dir():
        continue
    species_name = species_dir.name
    print(f"\n🔄 Processing species: {species_name}")

    # 1) Daten einlesen
    df = pd.read_csv(species_dir / "train.csv")

    # 2) Feature-Spalten auswählen
    # Für Eurasian Otter bleiben wir bei dist/ang/t…-Columns
    if species_name == "eurasian_otter_(1)":
        feature_cols = [
            col for col in df.columns
            if col.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[col])
        ]
    else:
        # alle numerischen ab 6. Spalte
        feature_cols = [
            col for col in df.columns[5:]
            if pd.api.types.is_numeric_dtype(df[col])
        ]
    print(f"  → {len(feature_cols)} numerische Features gefunden.")

    # 3) Paarvergleiche generieren (gleiche Einstellungen für alle Species)
    comparisons = generate_pairwise_comparisons_from_df(
        df,
        id_col="individual_id",
        group_sizes=[2,3,5,7],
        n_repeats=3,
        mode="both",
        selfmatch_factor=10
    )
    print(f"  → {len(comparisons)} raw pairs generated.")
   # comparisons =random.sample(comparisons, 20)
    
    # 4) Sweep nur über selection_methods
    species_results = []
    timing_log = []

    for sel in tqdm(selection_methods, desc=f"FS methods for {species_name}"):
        t0_sel = time.time()
        for use_sex in (False, True):
            tag = "sex_on" if use_sex else "sex_off"
            # Pfad zum Sex-Modell
            sexmodel_path = models_root / f"{species_name}.joblib" if use_sex else None

            with tqdm_joblib(tqdm(desc=f"Pairs ({sel} | {tag})", total=len(comparisons))):
                results = run_all_pairwise_projections_parallel(
                    comparisons=comparisons,
                    df=df,
                    feature_cols=feature_cols,
                    k_features=feature_counts,
                    reducers=reducers,
                    selection_method=sel,
                    n_components=n_components_list,
                    use_sexmodel_prediction=use_sex,
                    sexmodel_path=str(sexmodel_path) if sexmodel_path else None,
                    debug=False,
                    n_jobs=n_jobs
                )
            dur = time.time() - t0_sel
            print(f"    • {sel} | sex={use_sex}: {len(results)} rows in {dur:.1f}s")

            # DataFrame speichern
            df_res = pd.DataFrame(results)
            df_res["selection_method"]        = sel
            df_res["use_sexmodel_prediction"] = use_sex
            df_res["species"]                 = species_name
            fname = output_root / f"{species_name}_pairwise_{sel}_{tag}_{len(comparisons)}pairs.csv"
            df_res.to_csv(fname, index=False)

            species_results.append(df_res)
            timing_log.append({
                "species":                species_name,
                "selection_method":       sel,
                "use_sexmodel_prediction":use_sex,
                "duration_s":             dur,
                "n_results":              len(results)
            })

    # pro Species auch Gesamt-CSV
    if species_results:
        sp_full = pd.concat(species_results, ignore_index=True)
        sp_full.to_csv(output_root / f"{species_name}_all_results.csv", index=False)
        print(f"  ✔️ Wrote species-combined results for {species_name}")

    # Timing speichern
    pd.DataFrame(timing_log).to_csv(output_root / f"{species_name}_timing_log.csv", index=False)

    all_species_results.append(sp_full)

# %%  
# --- Globales Zusammenführen über alle Species ---
if all_species_results:
    all_full = pd.concat(all_species_results, ignore_index=True)
    all_full.to_csv(output_root / "all_species_all_results_combined.csv", index=False)
    print("\n🎉 Alle Species kombiniert gespeichert in 'all_species_all_results_combined.csv'")
